# Notebook 6 – AI Risk Score & Dashboard Export

In [ ]:
import pandas as pd
import numpy as np

summary = pd.read_excel("output/Portfolio_Analytics.xlsx")
pred = pd.read_csv("output/walkforward_predictions.csv", parse_dates=["Date"])
mc = pd.read_csv("output/monte_carlo_summary.csv")
weights = pd.read_csv("output/max_sharpe_weights.csv", header=None, names=["Stock","Weight"])

# Helper
def metric(name):
    return float(summary.loc[summary["Metric"]==name,"Value"].iloc[0])


## AI Risk Score

In [ ]:
vol = abs(metric("Annual Volatility"))
dd = abs(metric("Maximum Drawdown"))
var95 = abs(metric("VaR95"))
cvar95 = abs(metric("CVaR95"))
loss_prob = pred["Loss_Probability"].mean()

def normalize(x, low, high):
    x = max(low, min(high, x))
    return 100*(x-low)/(high-low)

risk_score = (
    0.25*normalize(vol,0.05,0.40) +
    0.20*normalize(dd,0.00,0.60) +
    0.15*normalize(var95,0.00,0.05) +
    0.15*normalize(cvar95,0.00,0.08) +
    0.25*(loss_prob*100)
)

risk_score = float(np.clip(risk_score,0,100))

if risk_score<30:
    level="Low"
elif risk_score<60:
    level="Moderate"
elif risk_score<80:
    level="High"
else:
    level="Very High"

print(risk_score, level)


## Executive Summary

In [ ]:
executive = pd.DataFrame({
    "KPI":[
        "AI Risk Score",
        "Risk Level",
        "Average Loss Probability",
        "Median Monte Carlo Ending Value",
        "Probability of Profit"
    ],
    "Value":[
        round(risk_score,2),
        level,
        round(loss_prob,4),
        mc.loc[mc.Metric=="Median Ending Value","Value"].iloc[0],
        mc.loc[mc.Metric=="Probability of Profit","Value"].iloc[0]
    ]
})
executive


## Power BI Export Tables

In [ ]:
dashboard_predictions = pred.copy()
dashboard_predictions["Return_Error"] = (
    dashboard_predictions["Actual_Return"] -
    dashboard_predictions["RF_Return"]
)

dashboard_predictions.to_csv(
    "output/pbi_predictions.csv",
    index=False
)

weights.to_csv(
    "output/pbi_weights.csv",
    index=False
)

executive.to_csv(
    "output/pbi_kpis.csv",
    index=False
)

summary.to_csv(
    "output/pbi_portfolio_metrics.csv",
    index=False
)


# Power BI Dashboard Layout

## Page 1 – Executive
- AI Risk Score (Gauge)
- Probability of Loss (Card)
- Portfolio CAGR
- Sharpe Ratio
- Max Drawdown
- Monte Carlo Median Value

## Page 2 – Portfolio
- Portfolio vs Benchmark
- Allocation Pie
- Weight Table

## Page 3 – Risk
- Rolling Volatility
- Drawdown
- VaR
- CVaR

## Page 4 – Machine Learning
- Actual vs Predicted Return
- Loss Probability
- Model Metrics

## Page 5 – Optimization
- Efficient Frontier
- Maximum Sharpe Weights
- Minimum Variance Weights

## Page 6 – Monte Carlo
- Distribution of Ending Portfolio Values
- Simulation Paths


# Resume Bullet

**Developed an AI-based Portfolio Risk & Return Analyzer using Python, Machine Learning, portfolio optimization, and Monte Carlo simulation. Implemented walk-forward time-series backtesting, predicted portfolio returns, volatility, and downside risk, evaluated performance using Sharpe Ratio, Sortino Ratio, VaR, CVaR, and Maximum Drawdown, and built Power BI-ready analytics datasets.**